# 10주 · 선형계·피벗·LU·조건과 잔차

수치해석및실습 CE273

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

학습 주제: 선형 연립방정식 (3) / 선형 연립방정식 (4)

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week10/CE273_W10.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week10)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE273-numerical-analysis/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 계산식 대조


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def linear_system(A, b):
    A=np.asarray(A,dtype=float);b=vector(b)
    if A.ndim!=2 or A.shape[0]!=A.shape[1] or A.shape[0]!=len(b) or not np.isfinite(A).all():
        raise ValueError("정방행렬·차원·유한한 값 확인 필요")
    x=np.linalg.solve(A,b)
    residual=A@x-b
    den=np.linalg.norm(A,np.inf)*np.linalg.norm(x,np.inf)+np.linalg.norm(b,np.inf)
    return {"x":x,"residual":residual,"backward_error":0. if den==0 else np.linalg.norm(residual,np.inf)/den,"condition_2":np.linalg.cond(A)}

def gaussian_pivot(A,b):
    A=np.asarray(A,dtype=float).copy();b=vector(b).copy()
    if A.ndim!=2 or A.shape!=(len(b),len(b)) or not np.isfinite(A).all():raise ValueError("정방행렬·우변 차원 확인 필요")
    n=len(b)
    for k in range(n):
        pivot=k+np.argmax(abs(A[k:,k]))
        if abs(A[pivot,k])<=np.finfo(float).eps*np.linalg.norm(A,np.inf):raise np.linalg.LinAlgError("0 또는 작은 피벗: 특이성·스케일 검토")
        A[[k,pivot]]=A[[pivot,k]];b[[k,pivot]]=b[[pivot,k]]
        for i in range(k+1,n):
            factor=A[i,k]/A[k,k];A[i,k:]-=factor*A[k,k:];b[i]-=factor*b[k]
    x=np.zeros(n)
    for i in range(n-1,-1,-1):x[i]=(b[i]-A[i,i+1:]@x[i+1:])/A[i,i]
    return x


<a id="s01"></a>

## 유일해·무한해·과소결정계 검산

**먼저 예측**: 식에 실제로 대입하면 제시된 해가 맞는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
A=np.array([[2.,-1],[-1,2]]);b=np.array([0.,3]);print(linear_system(A,b))
assert np.allclose(np.linalg.solve(A,b),[1,2])
# 10쪽 그림 y=x+1, y=-x+2에 일치하는 정정 행렬
Aplot=np.array([[-1.,1],[1,1]]);bplot=np.array([1.,2]);assert np.allclose(np.linalg.solve(Aplot,bplot),[.5,1.5])
B=np.array([[1.,1,1],[2,3,4]]);rhs=np.array([6.,15])
for t in [0.,2.]:
    x=np.array([3+t,3-2*t,t]);assert np.allclose(B@x,rhs)
print("과소결정계: x=3+t, y=3-2t, z=t")

**조건 변경**: 유일해·무한해·불일치 예제의 rank(A),rank([A|b]) 비교

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 원본 방정식·변수 순서·행렬의 차원·계수·대입 검산 명시

**결과 검토**: 원본 10쪽 중복 행렬과 18쪽 x의 상수를 정정. 그림·식·결과의 일치 확인

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


<a id="s02"></a>

## 피벗·LU와 Cholesky 조건

**먼저 예측**: 0 피벗과 행 교환, 특이행렬은 같은 개념인가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
A=np.array([[0.,1],[1,0]]);b=np.array([2.,1])
x=gaussian_pivot(A,b);P,L,U=linalg.lu(A)
assert np.allclose(A,P@L@U) and np.allclose(A@x,b)
print("피벗 포함 해:",x,"det:",np.linalg.det(A))
SPD=np.array([[2.,1],[1,2]]);C=np.linalg.cholesky(SPD);assert np.allclose(C@C.T,SPD)
try:np.linalg.cholesky(np.ones((2,2)))
except np.linalg.LinAlgError:print("준정부호만으로 일반 Cholesky 성공 보장 불가")
# 작은 잔차와 해 오차가 다를 수 있는 추가 실험
ill=np.array([[1.,1],[1,1+1e-8]]);rhs=ill@np.ones(2);changed=rhs+np.array([0,1e-8])
answer=linear_system(ill,changed);print("condition:",answer["condition_2"],"x:",answer["x"],"residual:",answer["residual"])

**조건 변경**: 작은 우변 교란과 조건수를 함께 확인

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: LU의 순열 규약 A=P@L@U·Cholesky의 대칭 양의 정부호·잔차·조건수 명시

**결과 검토**: 분해 결과는 API의 순열 규약으로 검산. 작은 잔차만으로 작은 해 오차 확정 불가

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

해 존재·유일성·피벗·분해 조건·조건수 구분

입력 항목:

- `그림_유일해_x`: 해당 계산값·항 수·오차 기준의 수치
- `그림_유일해_y`: 해당 계산값·항 수·오차 기준의 수치
- `과소결정_t0_x`: 해당 계산값·항 수·오차 기준의 수치
- `준정부호이면_항상_Cholesky_성공`: 판단 · Python `True`/`False`
- `작은_잔차가_항상_작은_해오차_보장`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인


In [ ]:
answers = {'그림_유일해_x': None, '그림_유일해_y': None, '과소결정_t0_x': None, '준정부호이면_항상_Cholesky_성공': None, '작은_잔차가_항상_작은_해오차_보장': None}


In [ ]:
def check_answers(values):
    expected = {'그림_유일해_x': 0.5, '그림_유일해_y': 1.5, '과소결정_t0_x': 3.0, '준정부호이면_항상_Cholesky_성공': False, '작은_잔차가_항상_작은_해오차_보장': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
